# 2. Joint construction: recurrence certificates and the common divisor

**Paper:** numerator sum presentation (Lemma 6.10 (`prop:z34-p-iv`)), dual sum presentation (Lemma 6.12 (`lem: dual sum`)), common divisor (Lemma 6.14 (`prop:z34-divisor`)), and the supporting arithmetic arguments in Appendix `app:zeta34-proof`.

Part A regenerates five exact telescoping certificates from the supplied matrices. Part B counts primes in the two presentations and certifies
\[
\liminf_{n\to\infty}\frac{\log\mathcal D_n}{n}\ge30.3467.
\]
Finite gcd comparisons are diagnostics. The general proof uses the symbolic recurrences, explicit boundary arguments, termwise valuation bounds, a complete floor-function arrangement and the prime number theorem.


Run every cell from top to bottom in a fresh Python kernel. This notebook uses only the local Python modules and matrix data included beside it; no other notebook needs to run first. Install the bundle's `requirements.txt` once. Saved output is a record of a completed run, while every certificate is recomputed when you run the cells.


In [1]:
from pathlib import Path
import sys
# Jupyter normally starts in the notebook folder; the bundle root also works.
_candidates = [Path.cwd(), Path.cwd() / "joint"]
NOTEBOOK_DIR = next((p.resolve() for p in _candidates
                     if (p / "certificates.py").is_file() and (p / "data" / "dual_companion.json").is_file()), None)
if NOTEBOOK_DIR is None:
    raise RuntimeError("Start this notebook from its joint/ folder or the bundle root.")
sys.path.insert(0, str(NOTEBOOK_DIR))
if hasattr(sys, "set_int_max_str_digits"):
    sys.set_int_max_str_digits(0)
print("Local joint-proof inputs found.")


Local joint-proof inputs found.


## A. Exact telescoping certificates

**Paper:** Lemma 6.9 (`prop:z34-q-iv`), Lemma 6.10 (`prop:z34-p-iv`), and Lemma 6.12 (`lem: dual sum`).

[certificates.py](certificates.py) constructs the five certificates using exact FLINT polynomials and independently substitutes each into its defining identity. The recurrences are derived from the supplied companion matrices. No recurrence is fitted to numerical sequence values.

The analytic transfer and boundary arguments below explain how the identities apply to the harmonic and Laurent readouts. These written mathematical steps are distinguished from the zero polynomial residuals verified by Python.


In [2]:
from pathlib import Path
from fractions import Fraction
import json
import sys

# Run with this folder as the working directory (the default in Jupyter).
if hasattr(sys, "set_int_max_str_digits"):
    sys.set_int_max_str_digits(0)
from certificates import (
    N, generate_all, prescribed_recurrence, forward_problem, dual_problem,
    apply_operator, degree_pair, evaluate_k, parse_rational,
    no_integer_roots_at_or_above,
)
from readouts import forward_values, dual_rows
DATA = NOTEBOOK_DIR / "data"
assert (DATA / "matrices_for_zeta3_4.json").is_file()
print("Exact polynomial arithmetic is ready.")

Exact polynomial arithmetic is ready.


For this notebook, the forward summand is
\[
B_{n,k}=\binom{k+12n}{10n}\binom{k+13n}{10n}\binom{k+14n}{10n}
\binom{15n}{n+k}\binom{15n}{k},
\]
and the readouts are $\mathbf q_n=\sum B_{n,k}$ and
$\mathbf p_{s,n}=\sum_{k=-n}^{15n}[B_{n,k}H^{(s)}_{4n+k}-B'_{n,k}H^{(s-1)}_{4n+k}/(s-1)]$.
The Python recurrence coefficient array `B` below has entries $B_0(n),\ldots,B_3(n)$; this local certificate notation is distinct from the two-index summand $B_{n,k}$. The certificate polynomial called `K` is likewise a scalar denominator-clearing factor, distinct from the paper's pairing matrix $K(n)$.


### A1. The prescribed recurrences

Our row companion convention is
\[
r_{n+3}=g_0(n)r_n+g_1(n)r_{n+1}+g_2(n)r_{n+2}.
\]
For the forward formulas, $r_n=12s_n/C_n$, where
\[
C_n=\frac{(12n)!(13n)!(15n)!}{(10n)!^3(2n)!(3n)!(4n)!n!}.
\]
We substitute this normalization and clear denominators and common polynomial factors. In the dual case no such normalization is applied. Both lists are in **descending shift order**:
\[
(Ls)(n)=B_0(n)s_{n+3}+B_1(n)s_{n+2}+B_2(n)s_{n+1}+B_3(n)s_n.
\]
The helper also verifies that the normalized coefficients reproduce the companion last column exactly.

In [3]:
# Reject a root at the first shifted integer, for every start used below.
for start in (0,1,2):
    assert not no_integer_roots_at_or_above(N-(start+1),start)
    assert not no_integer_roots_at_or_above(N-start,start)
    assert no_integer_roots_at_or_above(N+1,start)
B = prescribed_recurrence(DATA, "forward")
DB = prescribed_recurrence(DATA, "dual")
assert no_integer_roots_at_or_above(B[0])
assert no_integer_roots_at_or_above(DB[0])
print("Forward coefficient degrees:", [p.degree() for p in B])
print("Dual coefficient degrees:   ", [p.degree() for p in DB])
print("Both leading coefficients are nonzero for every integer n >= 0.")

Forward coefficient degrees: [231, 231, 231, 231]
Dual coefficient degrees:    [228, 228, 228, 228]
Both leading coefficients are nonzero for every integer n >= 0.


### A2. What the polynomial solver does

Each problem has the form
\[
A(n,k)P(n,k+1)-B_-(n,k)P(n,k)=F(n,k).
\]
Here $A$ and $B_-$ are monic of degree five in $k$. Their $k^4$ coefficients differ by $-d$, where $d=114$ for the forward certificate, $107$ for the dual scalar certificate, and $35$ for each dual row certificate.

For a proposed term $p_i(n)k^i$, its highest possible contribution is $(i-d)p_i(n)k^{i+4}$. We solve downward with nonzero **integer** pivots $i-d$. The leading coefficient $p_d(n)$ remains unknown; the last four equations determine it by exact division in $\mathbb Q[n]$.

Finally, a separate full polynomial substitution verifies the returned $P$. Failure to solve, nonzero division remainder, or nonzero substitution residual raises an exception.

### A3. Generate and verify all five certificates

For the forward summand $B_{n,k}$ from Lemma 6.9 (`prop:z34-q-iv`), put
\[
K=\prod_{r=1}^{30}(10n+r)^3,\qquad
D=\prod_{a=1}^{4}\prod_{r=1}^{3a}(k+an+r)
  \prod_{r=1}^{42}(14n-k+r)\prod_{r=1}^{45}(15n-k+r).
\]
Set $h=B_{n,k}/(KD)$. Then $h(n,k+1)/h(n,k)=A(n,k)/B_-(n,k+1)$, where
\[
\begin{aligned}
A&=(k+12n+1)(k+13n+1)(k+14n+1)(14n+42-k)(15n+45-k),\\
B_-&=k(k+n+3)(k+2n+6)(k+3n+9)(k+4n+12).
\end{aligned}
\]
The finite-product forcing is $F=\sum_{j=0}^{3}B_{3-j}(n)B_{n+j,k}/h(n,k)$. Consequently the checked polynomial identity gives
\[
LB=\Delta_k\bigl(B_-Ph\bigr).
\]
The dual term is
\[
T(n,k)=\frac{(-1)^k(16n+k)!(17n+k)!}
{(2n+k)!(n+k)!k!(11n-k)!(k-n)!(12n-k)!(k-2n)!(10n+1-k)!}.
\]
For $h=1$ or $3$, define
\[
\begin{aligned}
D_h&=\prod_{a=1}^{2}\prod_{r=1}^{ha}(k+an+r)
\prod_{b=11}^{12}\prod_{r=1}^{hb}(bn-k+r)
\prod_{r=1}^{10h}(10n+1-k+r),\\
A_h&=-(16n+k+1)(17n+k+1)(11n+11h-k)(12n+12h-k)(10n+1+10h-k),\\
B_h&=k(k-n)(k-2n)(k+n+h)(k+2n+2h).
\end{aligned}
\]
The scalar certificate uses $h=3$ and $F=\sum_{j=0}^3 DB_{3-j}T(n+j,k)/(T/D_3)$.
For column $j=0,1,2$, the row certificate uses $h=1$ and proves
\[
T(n+1,k)k^j-\sum_{i=0}^2Q_{ij}(n)k^iT(n,k)=\Delta_k V_j(n,k),
\quad V_j=\frac{B_1P_jT}{d_jD_1},
\]
where $d_j$ clears the denominators in column $j$ of $Q$. The three row certificates are essential: a first-coordinate recurrence alone would not identify the other coordinates.

In [4]:
cert = generate_all(DATA, progress=None)
for name, item in cert.items():
    if isinstance(item, dict):
        print(f"{name:12s}: exact substitution PASS; degree (n,k) = {item['degrees']}")
print("Generation and verification seconds:", cert["elapsed_seconds"])

forward     : exact substitution PASS; degree (n,k) = (433, 114)
dual_scalar : exact substitution PASS; degree (n,k) = (331, 107)
dual_row_0  : exact substitution PASS; degree (n,k) = (89, 35)
dual_row_1  : exact substitution PASS; degree (n,k) = (88, 35)
dual_row_2  : exact substitution PASS; degree (n,k) = (88, 35)
Generation and verification seconds: 2.36


The displayed degrees are calculated from the regenerated, expanded polynomial coefficients. Saved Maple degree metadata is not used. The exact substituted identity, with coefficients derived from the supplied JSON, is the verification criterion.

We can inspect any coefficient without printing a many-page polynomial. `cert["forward"]["P"][i]` is the coefficient of $k^i$ as a polynomial in $n$.

In [5]:
P = cert["forward"]["P"]
A, Bminus, F, D, K = forward_problem(B)
assert apply_operator(A, Bminus, P) == F
assert len(P) - 1 == 114
assert evaluate_k(Bminus, 0) == 0
assert all(c > 0 for c in K.coeffs())
print("Forward polynomial identity and k=0 boundary: PASS")

Forward polynomial identity and k=0 boundary: PASS


### A4. Forward summation and transfer to the harmonic forms

The factorial continuation of $h$ is
\[
h=\frac{(15n)!^2}{(10n+30)!^3}
\frac{(k+12n)!(k+13n)!(k+14n)!}
{(k+n+3)!(k+2n+6)!(k+3n+9)!(k+4n+12)!(14n+42-k)!k!(15n+45-k)!}.
\]
At $k=0$, $B_-=0$ and $h$ is finite. At $k=14n+43$, the reciprocal factorial with argument $-1$ vanishes and the other factors are finite. Summation therefore proves $L\mathbf q=0$ for all $n\ge0$.

For the harmonic forms, let
\[
\mathscr H_n(s)=(-1)^n\prod_{a=2}^4\frac{(s-(10+a)n)_{10n}}{(10n)!}
\frac{(15n)!}{(s-n)_{15n+1}}\frac{(15n)!}{(s)_{15n+1}}.
\]
Gamma reflection gives $\mathscr H_n(s)=\pi^2B_\Gamma(n,-s)/\sin^2(\pi s)$; its multiplier is 1-periodic. Its principal parts are
\[
\mathscr H_n(s)=\sum_{k=-n}^{15n}\left(\frac{B_{n,k}}{(s+k)^2}-\frac{B'_{n,k}}{s+k}\right).
\]
Here $B'_{n,k}$ is the Gamma-interpolation derivative, including the simple-zero tails. Hence
\[
\sum_{m>4n}\mathscr H'_n(m)=-2(\zeta(3)\mathbf q_n-\mathbf p_{3,n}),\qquad
\frac12\sum_{m>4n}\mathscr H''_n(m)=3(\zeta(4)\mathbf q_n-\mathbf p_{4,n}).
\]
The same verified certificate transfers to
\[
L\mathscr H_n(s)=\mathscr V_n(s+1)-\mathscr V_n(s),\quad
\mathscr V_n(s)=-\frac{B_-(n,1-s)P(n,1-s)\mathscr H_n(s-1)}{K(n)D(n,1-s)}.
\]
For $n\ge2$, rewrite every tail as a sum over integers $m\ge12n$. The triple-zero interval of $\mathscr H_z$ is $[4z+1,12z]$ for each shifted index $z=n,n+1,n+2,n+3$. Thus every omitted first and second derivative vanishes, and the included endpoint term at $m=12n$ also vanishes. Telescoping now has lower boundary $\mathscr V_n(12n+t)=O(t^3)$. At infinity the degree check gives $\mathscr V_n(s)=\text{constant}+O(s^{-1})$; the constant contributes neither a first nor a second Taylor coefficient. Differentiation of the tails is justified by uniform $O(m^{-3})$ and $O(m^{-4})$ bounds for fixed $n$.

It follows that $L\mathbf p_3=L\mathbf p_4=0$ for $n\ge2$, using $L\mathbf q=0$. The two remaining recurrence indices are checked exactly below.

In [6]:
# Nonvanishing of the lower-boundary denominator for every integer n >= 2.
assert no_integer_roots_at_or_above(evaluate_k(D, 1 - 12*N), start=2)
assert 5 + (len(P)-1) - (len(D)-1) - 2 <= 0
# Integer interval inequalities, after n=u+2, u>=0.
for difference in [12*N-(4*(N+3)+1), 12*N-1-(4*N+1)]:
    assert all(c >= 0 for c in difference(N+2).coeffs())

values = {n: forward_values(n) for n in range(5)}
for n in (0, 1):
    for coordinate in (1, 2):  # p3 and p4
        residual = sum(Fraction(str(B[i](n))) * values[n+3-i][coordinate]
                       for i in range(4))
        assert residual == 0
print("Forward boundaries and the four exceptional harmonic checks: PASS")

Forward boundaries and the four exceptional harmonic checks: PASS


### A5. Dual residue transfer and the exceptional first step

Let $R_n(s)$ be the paper’s dual rational kernel stated below, and $f_j(n,s)=(-s)^jR_n(s)$. Reflection gives
\[
T_\Gamma(n,-s)=e^{-i\pi s}\frac{\sin^3(\pi s)}{\pi^3}R_n(s).
\]
The inverse multiplier is 1-periodic. Each verified row certificate therefore becomes a rational identity
\[
f_j(n+1,s)-\sum_iQ_{ij}(n)f_i(n,s)=H_j(n,s+1)-H_j(n,s),
\]
\[
H_j(n,s)=-\frac{B_1(n,1-s)P_j(n,1-s)R_n(s-1)}{d_j(n)D_1(n,1-s)}.
\]
All its finite poles are integers. Summing each order of principal-part coefficient is invariant under translation $s\mapsto s+1$. This proves the full $a$ and $b$ row recurrences separately, without assuming linear independence of zeta values.

Because $f_j=O(s^{j-4})$, its simple-pole coefficients sum to zero. Partial fractions give
\[
-\sum_{m>2n}\partial_s f_j(n,m)=\mathbf a_{n,j}\zeta(4)+\mathbf b_{n,j}\zeta(3)+\mathbf d_{n,j}.
\]
For $n\ge1$, rewrite every tail as a sum over integers $m\ge16n$. All omitted terms and the included endpoint term lie in the double-zero interval $[2z+1,16z]$ of $R_z$, for $z=n,n+1$. Telescoping therefore has lower boundary $H_j(n,16n)$, whose derivative vanishes; the degree bound gives $H'_j=O(s^{-2})$ at infinity. Thus the differentiated row certificate sums to zero. Subtracting the already proved $a$ and $b$ recurrences proves the $d$ recurrence.

The exceptional $n=0$ step is a finite exact check. It includes the simple pole at $s=-1$, which must not be dropped.
The kernel is $R_n(s)=(-1)^n(1+2n-s)_{14n}(1+n-s)_{16n}/[(s)_{11n+1}(s+n)_{11n+1}(s+2n)_{8n+2}]$.


In [7]:
A1, B1, D1 = dual_problem(1)
assert no_integer_roots_at_or_above(evaluate_k(D1, 1 - 16*N), start=1)
for j in range(3):
    assert 5 + len(cert[f"dual_row_{j}"]["P"])-1 - (len(D1)-1) - 4 <= 0
for difference in [16*N-(2*N+3), 16*N-1-(2*N+1), 16*(N+1)-(16*N-1)]:
    assert all(c >= 0 for c in difference(N+1).coeffs())

initial = dual_rows(0)
assert initial == [[3,0,0], [-2,-2,0], [1,1,1]]
Qraw = json.loads((DATA / "dual_matrices_zeta3_4.json").read_text())["G"]
Q0 = []
for row in Qraw[:3]:
    row0 = []
    for entry in row[:3]:
        value = parse_rational(entry)
        row0.append(Fraction(str(value.p(0) / value.q(0))))
    Q0.append(row0)
next_row = [[sum(initial[r][i]*Q0[i][j] for i in range(3))
             for j in range(3)] for r in range(3)]
assert next_row == dual_rows(1)
print("Dual boundaries, all three initial rows and exceptional n=0 step: PASS")

Dual boundaries, all three initial rows and exceptional n=0 step: PASS


### A6. What has now been established

Together with the companion and initial-value identities in the matrix and sums notebook, these exact certificates and the analytic transfer arguments prove the forward scalar readouts and all three dual residue rows for every integer $n\ge0$.

No finite-depth numerical convergence test replaces a certificate here. Conversely, these certificates alone do not establish the final irrationality measures: those also require the arithmetic divisor and analytic growth estimates.

In [8]:
(NOTEBOOK_DIR / "results").mkdir(exist_ok=True)
report = {
    "all_five_polynomial_substitutions_exact": True,
    "forward_harmonic_exceptional_indices": [0, 1],
    "dual_exceptional_index": 0,
    "lower_and_infinite_boundaries_checked": True,
    "degrees_n_k": {name: list(item["degrees"]) for name,item in cert.items()
                    if isinstance(item,dict)},
    "elapsed_seconds": cert["elapsed_seconds"],
}
(NOTEBOOK_DIR / "results" / "telescoping_certificate.json").write_text(json.dumps(report, indent=2))
print("All required telescoping checks passed.")

All required telescoping checks passed.


## B. Prime counts and the common-divisor density — Lemma 6.14 (`prop:z34-divisor`)

The three normalized readouts are multiplied by $D_n=6d_{19n}^4$. We construct the divisor $\mathcal D_n$ from lower bounds for their prime valuations, choosing the better of the forward and dual presentations for each prime.

[density.py](density.py) uses Python integers and exact fractions only. Its complete floor geometry, offset handling and rational integration are included. No floating-point value enters the density certificate.

The identities and all-$n$ integrality argument are given in Notebook 1 and Part A above. The prime number theorem converts the valuation bounds into the asymptotic density; it is a mathematical input, not a finite computation.


In [9]:
from fractions import Fraction as F
from math import factorial, lcm
from density import (
    valuation, decimal_text, forward_term_bound, dual_pole_data,
    counts, counts_core, rational_grid, density_certificate,
    finite_prime_checks, exact_readouts, lambda_scalar,
    FORWARD_SLOPES, DUAL_SLOPES, LAMBDA_EXPONENTS,
)
print("Exact arithmetic ready: integers and fractions only.")

Exact arithmetic ready: integers and fractions only.


### B1. Legendre's formula and the two presentations

For a prime $p$,
\[
v_p(m!)=\sum_{j\ge1}\left\lfloor m/p^j\right\rfloor,
\qquad
v_p\!\left(\sum_k t_k\right)\ge\min_k v_p(t_k).
\]
If $p^2>30n$, only the first Legendre floor is needed for the factorials here. Harmonic numbers of order $s$ cost at most $s$ powers of $p$; differentiation costs at most one additional power.

The forward readouts are $\mathbf q_n,\mathbf p_{3,n},\mathbf p_{4,n}$; multiply them by $D_n=6\operatorname{lcm}(1,\ldots,19n)^4$. The second presentation expresses the same readouts as dual $3\times3$ minors with scalar
\[
\Lambda_n=\frac{(8n)!(9n)!^2(11n)!^3(12n)!^3(13n)!^2}
 {6(16n)!^2(17n)!^2(18n)!^2(19n)!}.
\]
There is **no residual $\Delta(n)^{-1}$** in this scalar. Its cancellation is essential.

At large primes, $v_p(\Lambda_n)$ is represented by $\varphi_\Lambda(x)=\sum_a e_a\lfloor ax\rfloor$, with $x=n/p$ and the exponents below.

In [10]:
def legendre_factorial(m, p):
    total = 0
    while m:
        m //= p
        total += m
    return total

for m in range(101):
    for p in (2, 3, 5, 7, 11, 13):
        assert legendre_factorial(m, p) == valuation(factorial(m), p)
print("Legendre's formula: exact checks passed.")
print("Lambda factorial exponents:", LAMBDA_EXPONENTS)
assert sum(a*e for a, e in LAMBDA_EXPONENTS.items()) == 0
print("The linear floor shifts in phi_Lambda cancel.")

Legendre's formula: exact checks passed.
Lambda factorial exponents: {8: 1, 9: 2, 11: 3, 12: 3, 13: 2, 16: -2, 17: -2, 18: -2, 19: -1}
The linear floor shifts in phi_Lambda cancel.


### B2. Exact finite checks — use the offsets

At finite $(n,p)$, use literal floors $\lfloor(an+bk+c)/p\rfloor$. Replacing these by limiting floors can give a wrong result near a breakpoint. The implementation keeps the $-2,-1,+1$ offsets and the true pole endpoint $10n+1$.

For each $n=4,\ldots,8$, we calculate the three readouts from their exact forward sums, verify that $D_n$ clears their denominators, and calculate the actual gcd. For every prime $\sqrt{30n}<p<31n$, both floor bounds must be at most its true valuation.

The dual bound is $v_p(D_n)+v_p(\Lambda_n)+\min(m_a+m_b,m_a+m_d,m_b+m_d)$, truncated below at zero. Each $m$ is the minimum over the appropriate dual summand bounds. The list called “dual captures” shows where the dual bound reaches the actual gcd valuation and improves the forward bound.

**Why these are bounds for every $n$, not just the checked values.** Assume $p>3$ and $p^2>30n$. Every relevant factorial argument and nonzero difference between poles or roots is at most $30n$. Legendre's formula then has one level. At a pole of order $m\le3$ with leading-coefficient valuation $\nu$, the Laurent coefficient of order $i$ has valuation at least $\nu-(m-i)$: expanding the regular factors to order $m-i$ introduces at most that many reciprocal differences. Multiplication by $(-s)^j$, $j\le2$, has integer coefficients and cannot worsen this bound. The harmonic factor $H_{2n+j}^{(i+1)}$ costs at most $(i+1)\mathbf1_{p\le2n+j}$ powers of $p$. These are precisely the layers minimized in `dual_pole_data`; the coefficient rows $a,b$ use the third and second layers. Products in a minor add the row bounds, and its integer third row introduces no loss.

For an interior forward term, $v_p(B'_{n,k})\ge v_p(B_{n,k})-1$ follows from its harmonic logarithmic derivative. The two boundary ranges use the explicit factorial ratios, including their offsets. The factors $1/2$, $1/3$ are $p$-adic units. Add the exact $v_p(\Lambda_n)$ and $v_p(D_n)=4\mathbf1_{p\le19n}$ as appropriate. All three $D_n$-scaled readouts are integers by the forward denominator-clearing argument, so zero is also a valid lower bound. This proves the finite floor bounds for all $n\ge1$; the 181 comparisons below are independent diagnostics.


In [11]:
finite_rows, captures = finite_prime_checks()
print("n  prime cases  dual captures")
for n in range(4, 9):
    cases = [r for r in finite_rows if r[0] == n]
    print(n, str(len(cases)).rjust(11), captures[n])
assert len(finite_rows) == 181
print("All 181 prime cases passed; all five scaled triples are integral.")
print("\nExample rows at n=4: n, p, forward, dual, actual")
for row in finite_rows:
    if row[0] == 4 and row[1] in (13, 41, 43, 47):
        print(row)

# Construct the divisor from the floor bounds, without using the actual gcd.
from density import certified_divisor
N = 4
divisor_N, prime_powers = certified_divisor(N)
D_N = 6*lcm(*range(1, 19*N+1))**4
integer_triple = [D_N*v/divisor_N for v in exact_readouts(N)]
assert all(v.denominator == 1 for v in integer_triple)
print("Certified divisor prime powers at N=4:", prime_powers)
print("Normalized integer digit counts:", [len(str(abs(v.numerator))) for v in integer_triple])

n  prime cases  dual captures
4          26 [43]
5          31 [13, 43, 47, 53]
6          36 [53, 59]
7          41 [59, 61, 67, 73]
8          47 [29, 67, 71, 79, 83]
All 181 prime cases passed; all five scaled triples are integral.

Example rows at n=4: n, p, forward, dual, actual
(4, 13, 0, 0, 1)
(4, 41, 3, 2, 4)
(4, 43, 3, 4, 4)
(4, 47, 3, 2, 3)
Certified divisor prime powers at N=4: {11: 2, 17: 1, 19: 1, 23: 2, 29: 1, 41: 3, 43: 4, 47: 3, 53: 1, 59: 1, 67: 1, 71: 2, 73: 3}
Normalized integer digit counts: [201, 201, 201]


### B3. The step functions and complete rational cells

Let $\gamma_F(x)$ and $\gamma_D(x)$ be the respective lower bounds. We may choose the stronger presentation independently for each prime:
\[
\gamma_{\mathrm{best}}(x)=\max\{\gamma_F(x),\gamma_D(x)\}.
\]
The minima include every floor breakpoint in the summation variable, the structural endpoints, and a midpoint in every interval between them. On each structural/harmonic band the factorial valuation is periodic in that variable with period one, so one complete period and the actual endpoints suffice.

**Why the integration cells are complete.** The relevant lines have form $y=m+s x$, with integer intercepts. The forward slopes lie in $[-14,15]$ and the dual slopes in $[-17,12]$; crossings within either minimization therefore have denominators at most 29. Pure floor jumps and harmonic cutoffs have denominators at most 30. All reduced fractions of denominator at most 40 give a safe refinement. Between consecutive fractions the line ordering, candidate floor values, and minima cannot change. Three-point comparisons below are additional diagnostics of this arrangement argument.

For $x>1$, all harmonic indicators are one and every structural band contains a whole period. The integer shifts caused by $x\mapsto x+1$ cancel in the factorial ratios and $\varphi_\Lambda$, proving period one. The program also compares unreduced and reduced evaluations, including boundary points.

**Thin shifted walls require extra care.** Staying away from an $x$-cell endpoint does not by itself remove the summation indices just beside a $y$-wall. For $x=n/p$, every unshifted wall $y=m+s x$ is the integer index $k_0=mp+sn$, because $m,s$ are integers. All factorial offsets satisfy $|c|\le2$. Therefore the integers $k_0+\delta$, $\delta=-3,-2,\ldots,3$, include every state at a shifted jump and on each side. Outside that neighborhood the ordinary open-band value applies. Structural endpoints, harmonic cutoffs, and the special pole $j=10n+1$ are included in the same enumeration; out-of-support indices are discarded.

The next certificate expands **every** wall and endpoint by these seven offsets on all 980 open cells in $(0,2)$. It proves that the expanded minima equal the counts already used. Within a cell the unshifted line ordering is fixed; coincident-line offsets have fixed integer order. Thus every local state is constant there, and one rational representative certifies the entire cell, rather than a sample of integer $n$ values. Periodicity handles the remaining unit blocks.

Implementation detail: at a representative $x=a/b$, each candidate has $y=Y/(2b)$. The integer model $n=2aT$, $p=2bT$, $k=YT+\delta$ with $T=16$ evaluates these states using literal floors. This model $p$ need not be prime: it only evaluates floor expressions. Every nonzero unshifted distance is at least $T$, while $|c\pm\delta|\le5<T$, so the calculation is exactly the chamber-limit calculation.


In [12]:
print("Forward slopes:", FORWARD_SLOPES)
print("Dual slopes:   ", DUAL_SLOPES)
for slopes in (FORWARD_SLOPES, DUAL_SLOPES):
    assert max(slopes)-min(slopes) <= 40
for x in (F(1,40), F(1,19), F(1,10), F(2,17), F(1), F(7,3), F(40)):
    f, d = counts(x)
    print(f"x={str(x):>5}: forward={f}, dual={d}, best={max(f,d)}")
    assert counts_core(x) == counts(x)
print("All displayed periodic checks passed.")

Forward slopes: (-14, -13, -12, -4, -3, -2, -1, 0, 14, 15)
Dual slopes:    (-17, -16, -2, -1, 0, 1, 2, 10, 11, 12)
x= 1/40: forward=0, dual=0, best=0
x= 1/19: forward=3, dual=2, best=3
x= 1/10: forward=0, dual=2, best=2
x= 2/17: forward=0, dual=1, best=1
x=    1: forward=0, dual=0, best=0
x=  7/3: forward=2, dual=2, best=2
x=   40: forward=0, dual=0, best=0
All displayed periodic checks passed.


In [13]:
from density import verify_offset_chambers
offset_certificate = verify_offset_chambers()
assert offset_certificate["checked_chambers"] == 980
print("PASS:", offset_certificate["checked_chambers"],
      "complete chambers, all integer offsets", offset_certificate["integer_offsets"])
print("The expanded wall-state minima equal the certified counts on every chamber.")

PASS: 980 complete chambers, all integer offsets [-3, -2, -1, 0, 1, 2, 3]
The expanded wall-state minima equal the certified counts on every chamber.


### B4. Exact density certificate

By the prime number theorem, a step exponent contributes density
\[
I=\int_0^\infty\frac{\gamma(x)}{x^2}\,dx.
\]
On a cell $(a,b)$ its contribution is exactly $\gamma((a+b)/2)(1/a-1/b)$. Both counts vanish on the first cell adjacent to zero.

We integrate through $X=40$ using **19,600 rational cells**. There are only two distinct blocks to evaluate because the period is one beyond $x=1$. We then bound each tail by $\sup\gamma/X$; the supremum includes open cells **and endpoints**.

The paper’s rounded bounds give the threshold $76+65.4516-112.8075=28.6441$. This notebook treats that number as an exact rational comparison target. Notebook 3 certifies the spectral and growth inequalities separately.

**From finite valuations to a divisor density: a fixed-neighborhood argument.** Define $\mathcal D_n$ using the literal finite bounds, with primes $p>3$, $p^2>30n$, $p<31n$, as in `certified_divisor`. Every prime power in this product divides all three normalized integers by the preceding all-$n$ bounds. The initial chamber checks also certify that the limiting counts vanish for $0<x<1/31$, so the prime cutoff discards no positive-density contribution.

Fix a finite cutoff $X$ and remove an $\varepsilon$-neighborhood of each rational $x$-cell endpoint. There are finitely many remaining closed intervals, all bounded away from zero wherever the count is positive. On these intervals distinct unshifted $y$-walls are uniformly separated. For sufficiently large $n$, primes with $x=n/p$ there satisfy $p\asymp n$ and hence $p^2>30n$; the shifts $c/p$ cannot change the wall ordering. The complete seven-offset certificate handles every thin-wall state, including shifted structural endpoints. Consequently the finite exponent is at least $\gamma_{\rm best}(n/p)$ uniformly on those intervals.

For a cell $(a,b)$ with constant exponent $v$, the prime number theorem gives contribution $v(1/(a+\varepsilon)-1/(b-\varepsilon))$ from its trimmed interval. Sum these finitely many lower bounds, then let $\varepsilon\downarrow0$. The first cell has exponent zero, so there is no singular contribution at zero. It follows that
\[
\liminf_{n\to\infty}\frac{\log\mathcal D_n}{n}
\ge\int_0^X\frac{\gamma_{\rm best}(x)}{x^2}\,dx.
\]
This argument does not require enumerating exceptional primes or claiming that a finite list of divisibilities exhausts them. It simply discards the neighborhoods for each fixed $\varepsilon$, proves the bound, and then takes the limit. Letting $X$ increase gives the full integral as well. The finite cutoff $X=40$ already suffices for linear independence.


In [14]:
certificate = density_certificate(cutoff=40, order=40, diagnostics=True)
I_F, I_D, I_best = certificate["integrals"]
TARGET = F("28.6441")       # 76 + 65.4516 - 112.8075, exactly
PAPER_DENSITY = F("30.3467")
assert certificate["cells"] == 19600
assert certificate["maxima"] == [3, 4]
assert certificate["individual_upper"][0] < F("26.29")
assert certificate["individual_upper"][1] < F("18.25")
assert I_best > PAPER_DENSITY > TARGET
assert I_best - TARGET > F("1.70")
print("Exact integration cells:", certificate["cells"])
for label, value in zip(("forward", "dual", "best per prime"), certificate["integrals"]):
    print(f"Integral through 40, {label:14}: {decimal_text(value)}")
print("Forward full integral <=", decimal_text(certificate["individual_upper"][0]), "< 26.29")
print("Dual full integral    <=", decimal_text(certificate["individual_upper"][1]), "< 18.25")
print("Combined truncated density > 30.3467; required threshold = 28.6441")
print("Margin above the paper threshold:", decimal_text(I_best-TARGET))


Exact integration cells: 19600
Integral through 40, forward       : 26.208798832530
Integral through 40, dual          : 18.142839628527
Integral through 40, best per prime: 30.346777819491
Forward full integral <= 26.283798832530 < 26.29
Dual full integral    <= 18.242839628527 < 18.25
Combined truncated density > 30.3467; required threshold = 28.6441
Margin above the paper threshold: 1.702677819491


The individual full integrals are too small for this target, even after including their entire tails. The **combined truncated integral already exceeds the target**. No estimate of its remaining nonnegative tail is required.

The values are recomputed from the floor functions. The next cell saves the exact rational values, every integration cell, and all finite prime checks for inspection. These certificate comparisons use exact rational arithmetic.

In [15]:
import csv, json
out = NOTEBOOK_DIR / "results"
out.mkdir(exist_ok=True)
summary = {
    "paper_label": "prop:z34-divisor",
    "arithmetic_status": "PASS",
    "scope": "Exact floor and density certificates. The all-index valuation arguments and PNT are stated in markdown.",
    "cutoff": certificate["cutoff"], "cells": certificate["cells"],
    "integrals_exact": [str(v) for v in certificate["integrals"]],
    "integrals_decimal_truncated": [decimal_text(v) for v in certificate["integrals"]],
    "periodic_maxima": certificate["maxima"],
    "tail_upper_exact": [str(v) for v in certificate["tail_upper"]],
    "individual_upper_exact": [str(v) for v in certificate["individual_upper"]],
    "paper_threshold_exact": str(TARGET), "paper_density_lower_exact": str(PAPER_DENSITY),
    "combined_margin_exact": str(I_best-TARGET),
    "finite_prime_cases": len(finite_rows),
    "complete_offset_chamber_certificate": offset_certificate,
    "dual_captures": captures,
}
(out / "density_certificate.json").write_text(json.dumps(summary, indent=2)+"\n")
with (out / "density_cells.csv").open("w", newline="") as stream:
    writer = csv.writer(stream)
    writer.writerow(("left", "right", "forward", "dual", "exact_weight"))
    writer.writerows(certificate["rows"])
with (out / "density_finite_primes.csv").open("w", newline="") as stream:
    writer = csv.writer(stream)
    writer.writerow(("n", "p", "forward", "dual", "actual"))
    writer.writerows(finite_rows)
assert summary["integrals_decimal_truncated"] == [
    "26.208798832530", "18.142839628527", "30.346777819491"]
print("Saved exact fractions, 19,600 cells and 181 prime diagnostics to results/.")


Saved exact fractions, 19,600 cells and 181 prime diagnostics to results/.


### B5. Interpretation and proof scope

- **Exact computation:** the conservative forward/dual counts give $I_F(40)=26.208798832530\ldots$, $I_D(40)=18.142839628527\ldots$, and $I_{\rm best}(40)=30.346777819491\ldots>28.6441$.
- **Exact tail bounds:** $I_F(\infty)\le I_F(40)+3/40$ and $I_D(\infty)\le I_D(40)+1/10$, both below the target.
- **Complete boundary certificate:** all 980 rational chambers include every integer offset from $-3$ through $3$, including the shifted pole endpoint $10n+1$. Expanded minima agree with the integrated counts.
- **Finite independent diagnostics:** exact readout gcds dominate the finite forward/dual bounds in all 181 checked prime cases. The limiting counts must not be substituted for these finite counts near shifted boundaries.
- **All-$n$ conclusion:** after the readout identities, denominator clearing, large-prime valuation lemmas, the complete offset-state certificate, the fixed-neighborhood PNT passage, and spectral inequalities are established, the combined divisor is strong enough for the theorem. This notebook certifies the arithmetic computation in that argument; it does not infer those mathematical statements from a finite experiment.

Source provenance: the user-supplied joint proof archive and the current manuscript section. The complete computational implementation is included in `density.py`; execution requires no Maple files.